# 🔎 Teste Inicial de Consumo da API Pública da DEV Community

## Origem dos Dados

A DEV Community é uma comunidade com mais de três milhões de desenvolvedores de software [1]. No site, os desenvolvedores escrevem artigos (posts de blog, perguntas em fóruns de discussão, tópicos de ajuda, etc.), participam de discussões e constroem seus perfis profissionais [2].

O site é hospedado pelo FOREM [2], um software de código aberto para a construção de comunidades. Na documentação do FOREM é disponibilizada uma API pública [3] que retorna os textos publicados, tags sobre o assunto tratado, entre outras informações. A documentação da API não estabelece uma licença específica para os dados disponibilizados. Os dados que podem identificar de alguma forma os autores dos textos foram ocultados.

## Objetivo

Este notebook tem como objetivo realizar um teste inicial de consumo da versão 1 [4] da API pública do DEV Community, avaliando o acesso aos endpoints disponíveis, a estrutura dos dados retornados e a viabilidade de utilização dessas informações no Hackathon ONE - Projeto TechMind para treinamento e avaliação do modelo de classificação de textos.

## Teste

In [35]:
# Importando bibliotecas
import requests
import pandas as pd
import time

from IPython.display import display
pd.reset_option('display.max_colwidth')

In [36]:
# Definindo categorias e suas tags correspondentes no DEV.to
categorias = {
    "Backend": "backend",
    "Frontend": "frontend",
    "Data Science": "datascience",
    "IA": "ai"
}


In [37]:
# Buscar artigos por categoria
artigos_dataset = []

for categoria, tag in categorias.items():
    print(f"Buscando artigos de {categoria}...")

    response = requests.get(
        "https://dev.to/api/articles",
        params={
            "tag": tag,
            "per_page": 5
        }
    )
    artigos = response.json()

    # Buscar o conteúdo completo (texto) de cada artigo via id
    for artigo in artigos:
        artigo_id = artigo["id"]

        response_detalhe = requests.get(
            f"https://dev.to/api/articles/{artigo_id}"
        )

        detalhe = response_detalhe.json()

        # filtrando informações do retorno
        artigos_dataset.append({
            "categoria": categoria,
            "titulo": detalhe["title"],
            "descricao": artigo["description"],
            "texto": detalhe.get("body_markdown", ""),
            "tags": detalhe["tag_list"]
        })

        # evitar muitas requisições seguidas
        time.sleep(0.5)
print("Dados obtidos")

Buscando artigos de Backend...
Buscando artigos de Frontend...
Buscando artigos de Data Science...
Buscando artigos de IA...
Dados obtidos


In [38]:
# Criar DataFrame
df = pd.DataFrame(artigos_dataset)

df.head(1)

,categoria,titulo,descricao,texto,tags
0,Backend,Webhook idempotency for stablecoin checkout,A stablecoin checkout is usually built around ...,A stablecoin checkout is usually built around ...,"architecture, backend, blockchain, fintech"


## Análise Exploratória

In [39]:
# Confirmando estrutura gerada e quantidades
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   categoria  20 non-null     object
 1   titulo     20 non-null     object
 2   descricao  20 non-null     object
 3   texto      20 non-null     object
 4   tags       20 non-null     object
dtypes: object(5)
memory usage: 932.0+ bytes


In [40]:
# Validando quantidade por categoria
df['categoria'].value_counts()

categoria
Backend         5
Frontend        5
Data Science    5
IA              5
Name: count, dtype: int64

In [41]:
# Analisando dados
df.head()

,categoria,titulo,descricao,texto,tags
0,Backend,Webhook idempotency for stablecoin checkout,A stablecoin checkout is usually built around ...,A stablecoin checkout is usually built around ...,"architecture, backend, blockchain, fintech"
1,Backend,Multi-Tenancy And The Concepts Behind It.,Introduction Building a multi-tenant applica...,\n\n## **Introduction**\n\nBuilding a multi-te...,"backend, architecture, webdev"
2,Backend,Monitoring Python RQ jobs: what to watch and h...,RQ (Redis Queue) is a delightfully simple way ...,[RQ (Redis Queue)](https://python-rq.org/) is ...,"python, redis, monitoring, backend"
3,Backend,"An 8,000 trade settlement simulator shows wher...","Java's Project Loom promised a ""free lunch"": s...","\n\nJava's Project Loom promised a ""free lunch...","java, jvm, backend"
4,Backend,"Build a Real-Time Dashboard with Python, FastA...",Your Dashboard Is Lying to You (And Polling Is...,# Your Dashboard Is Lying to You (And Polling ...,"streaming, datapipeline, backend"


In [42]:
# Verificando categoria e texto
df[['categoria', 'texto']]

,categoria,texto
0,Backend,A stablecoin checkout is usually built around ...
1,Backend,\n\n## **Introduction**\n\nBuilding a multi-te...
2,Backend,[RQ (Redis Queue)](https://python-rq.org/) is ...
3,Backend,"\n\nJava's Project Loom promised a ""free lunch..."
4,Backend,# Your Dashboard Is Lying to You (And Polling ...
5,Frontend,{% embed https://primeui.dev/nextchapter minim...
6,Frontend,# Understanding React DOM: The Bridge Between ...
7,Frontend,\n\n如是我闻。一时，佛在瓦特纳（W3C）大达摩院，与大比丘众、诸大架构师、无量程序员俱。...
8,Frontend,{% embed https://dev.to/olboyarshinova/one-com...
9,Frontend,CSS variables are easy to create.\n\nA maintai...


In [43]:
# Verificando lista de tags associadas
df[['categoria', 'tags']]

,categoria,tags
0,Backend,"architecture, backend, blockchain, fintech"
1,Backend,"backend, architecture, webdev"
2,Backend,"python, redis, monitoring, backend"
3,Backend,"java, jvm, backend"
4,Backend,"streaming, datapipeline, backend"
5,Frontend,"frontend, news, opensource, webdev"
6,Frontend,"frontend, javascript, react, webdev"
7,Frontend,"frontend, javascript, typescript, webdev"
8,Frontend,"a11y, frontend, testing, webdev"
9,Frontend,"css, frontend, webdev, design"


In [44]:
# Analisando um texto completo - Backend
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Backend'][['descricao', 'texto', 'tags']].iloc[[0]])

,descricao,texto,tags
0,A stablecoin checkout is usually built around one important event: a transaction was detected and the...,"A stablecoin checkout is usually built around one important event: a transaction was detected and the merchant order can move forward.\n\nThat sounds simple until retries, duplicate callbacks, delayed confirmations, and partial payments show up.\n\nIf your webhook handler is not idempotent, the same blockchain event can accidentally create two business-side effects:\n\n- two paid-order transitions\n- two emails\n- two license activations\n- two shipment requests\n- two accounting records\n\nThe blockchain transaction is final. Your application state still needs protection.\n\n## A practical webhook rule\n\nTreat every payment event as an input to a state machine, not as a command to blindly mark an order as paid.\n\nFor each incoming webhook, I like to check four things before changing merchant state:\n\n1. Is the event signature valid?\n2. Have we already processed this event id or transaction hash?\n3. Is the target payment intent still in a state that can change?\n4. Does the observed payment match the expected amount, token, chain, and expiry window?\n\nOnly after those checks should the order move to a new state.\n\n## Example shape\n\nA minimal handler can look like this:\n\n```js\nasync function handlePaymentWebhook(event) {\n assertValidSignature(event)\n\n const alreadyProcessed = await db.webhookEvents.find(event.id)\n if (alreadyProcessed) return { ok: true }\n\n await db.transaction(async (tx) => {\n await tx.webhookEvents.insert({ id: event.id, txHash: event.txHash })\n\n const payment = await tx.paymentIntents.findForUpdate(event.paymentIntentId)\n if (!payment) return\n\n if (![""pending"", ""underpaid""].includes(payment.status)) return\n\n const nextStatus = classifyPayment(payment, event)\n\n if (nextStatus !== payment.status) {\n await tx.paymentIntents.update(payment.id, { status: nextStatus })\n await tx.orderEvents.insert({ orderId: payment.orderId, type: nextStatus })\n }\n })\n\n return { ok: true }\n}\n```\n\nThe important part is not the exact code. It is the constraint: the event record and the order transition should be committed together.\n\nIf the process crashes halfway through, retrying the webhook should either complete the transition once or safely return without repeating side effects.\n\n## States worth modeling early\n\nFor USDT or USDC checkout, I would avoid only having `pending` and `paid`.\n\nAt minimum, model:\n\n- pending\n- paid\n- expired\n- underpaid\n- overpaid\n- wrong_network\n- manual_review\n\nThose states make support and merchant dashboards much clearer. They also prevent your code from pretending every payment problem is a binary success/failure case.\n\n## Why I am thinking about this\n\nI am building ChainPay, a stablecoin checkout for merchants, WooCommerce stores, SaaS apps, Telegram sellers, and API workflows.\n\nThe goal is not just to display a QR code. The harder part is making payment state boring and predictable for the merchant.\n\nDocs:\nhttps://chainpay.to/docs?utm_source=devto&utm_medium=article&utm_campaign=webhook-idempotency-2026-07\n\nIf you have implemented payment webhooks before, what is the failure case you always design for first?","architecture, backend, blockchain, fintech"


In [45]:
# Analisando um texto completo - Frontend
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Frontend'][['descricao', 'texto', 'tags']].iloc[[0]])

,descricao,texto,tags
5,...,{% embed https://primeui.dev/nextchapter minimal %},"frontend, news, opensource, webdev"


In [46]:
# Analisando um texto completo - Data Science
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'Data Science'][['descricao', 'texto', 'tags']].iloc[[0]])

descricao  \
10  📊 Originally published on InsightRaider — a data platform tracking 152,362 active Gumroad...   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [47]:
# Analisando um texto completo -  IA
with pd.option_context('display.max_colwidth', None):
  display(df[df['categoria'] == 'IA'][['descricao', 'texto', 'tags']].iloc[[0]])


,descricao,texto,tags
15,"A post on X last week: ""The fact that people can't even reply to posts without AI anymore says a...","A post on X last week:\n\n> ""The fact that people can't even reply to posts without AI anymore says a lot more about them than they think.""\n\nThe replies agreed. Nobody asked the obvious question: says what, exactly?\n\n---\n\nHere's what I think it says: nothing new.\n\nPeople have been hollow online since the forum era. ""So true!!!"" didn't become meaningless when GPT launched. It was meaningless in 2009. Copy-pasted under blog posts. Typed by hand. Fully human. Fully empty.\n\nWhat changed isn't the emptiness. What changed is the cost to produce it.\n\nThat's a different thing.\n\n---\n\nAI is reportedly writing and reading more of the internet than people are at this point. I haven't chased down the study, but it tracks with what everyone's noticing.\n\nBut the conclusion most people draw from that — that authenticity is dying — assumes there was a lot of it before. There wasn't. There was friction. Friction isn't the same as authenticity. It just made the emptiness more expensive to ship.\n\nAutocorrect didn't make people bad texters. It made bad texters faster. The badness was already there.\n\n---\n\nThe tell is whether anyone is actually home behind it. The tool doesn't matter.\n\nSomeone typing ""this resonated with me 🙏"" isn't more present than someone who generated it. They just did more work to say nothing. That's not a virtue.\n\nIt does cost something. It costs attention. It costs the willingness to say something specific, something you actually think, something that could be wrong. Most people online weren't paying that cost before AI, and they're not paying it now.\n\n---\n\nThe gap was never intelligence. It was performance. It just wasn't visible when the performance required typing.\n\n---\n\nGeneration got cheap. We never built good filters for the expensive stuff — actual thinking, actual specificity, actual stakes.\n\nSlop has always existed. In dev work, in blog posts, in comments. The skill was always knowing where slop belongs and when to clean it up. Fast parallel experimentation? Slop is fine. Shipping to production without understanding it? That's the problem.\n\nSame principle applies to replies.\n\nUse the tool. Own what comes out.\n\nThe people performing outrage about AI replies are doing the same thing. Pattern-matching to a moment. Not thinking it through.\n\n---\n\n*AI helped me research, structure, and edit this piece. The arguments, the examples, and the opinions are mine. So is whatever's wrong with them.*","ai, watercooler, discuss"


## Primeiros Insights


* A coluna descrição não é um resumo. Seria somente um trecho do texto completo

* Conforme documentação da API e análise superficial dos textos, pode haver post de blog com conteúdo técnico mas também pode haver outros tipos de texto como comentários sobre um post ou somente um embed (um link que foi compartilhado em alguma postagem), por exemplo.  

  Isso sugere a necessidade de definir alguns parâmetros de limpeza dos dados, como excluir links e/ou manter somente registros com textos acima de uma determinada quantidade de caracteres.

* Os textos tem mais de uma lingua, necessário filtrar somente inglês para treino

* Aparentemente as tags representam os assuntos do texto.

* Há textos com emoji

## Conclusão e Recomendações

* Não houve problemas no acesso da API. Dados retornados conforme informados na documentação

* Até o momento não foi localizada informação clara sobre a licença de utilização dos dados

* É necessário uma avaliação mais profunda dos dados

* Recomendações: 
    * Obter uma quantidade maior de dados
    * Separar amostras por categoria para avaliar a qualidade dos dados, visando o objetivo do projeto. 
    * Pesquisar sobre utilização dos dados 

## Referências



[1] DEV COMMUNITY. *Site*. Disponível em: <https://dev.to/>. Acesso em: 9 jul. 2026.

[2] FOREM. *Sobre dev.to*. Disponível em: <https://github.com/forem/forem/>. Acesso em: 9 jul. 2026.

[3] API. *Versões da API*. Disponível em: <https://developers.forem.com/api>. Acesso em: 9 jul. 2026.

[4] Forem API V1. *Documentação da API*. Disponível em: <https://developers.forem.com/api/v1>. Acesso em: 9 jul. 2026.